# Multi-step rainfall forecasting – Kandi (Benin)

Code of the article *Impact of Training Data Size and Multi-Step Output Strategies on AI-Based Dekadal and Monthly Rainfall Forecasting* (PLOS ONE, 2026).

Put `end_kandi.csv` and `Data_Kandi_end.csv` in the folder given by `DATA_DIR` below, then run the cells in order (*Run → Run All Cells*).

In [ ]:
# Folders (change if needed)
DATA_DIR = "data"
OUT_DIR = "results"

## 1. Experimental design

In [ ]:
"""Experimental design: datasets, historical-data configurations and constants.

Every number in this module is taken from the Methods section of the article.
Changing a value here changes the experiment; nothing is hard-coded elsewhere.
"""


from dataclasses import dataclass

SEED = 42
HORIZON = 3                    # forecast steps T+1, T+2, T+3
DIRMO_BLOCKS = ((0, 2), (2, 3))  # block 1 = {T+1, T+2}, block 2 = {T+3} (0-based, end-exclusive)
TSCV_SPLITS = 5                # folds of TimeSeriesSplit used for hyperparameter tuning
KNN_NEIGHBORS = 5              # KNN imputation of missing values (k = 5)

MODELS = ("SVR", "RF", "CatBoost", "ANN", "LSTM", "TNN")
STRATEGIES = ("Direct", "Recursive", "MIMO", "DIRMO")


@dataclass(frozen=True)
class DatasetSpec:
    """Description of one input file (one temporal resolution)."""

    name: str
    filename: str
    target: str
    features: tuple[str, ...]
    year_column: str
    look_back: int
    unit: str
    sep: str = ";"
    decimal: str = "."

    @property
    def inputs(self) -> tuple[str, ...]:
        """Model inputs: lagged rainfall first, then the exogenous predictors.

        The target must stay in column 0: the Recursive strategy writes its
        own forecasts back into that column.
        """
        return (self.target, *self.features)


DATASETS: dict[str, DatasetSpec] = {
    "dekadal": DatasetSpec(
        name="dekadal",
        filename="end_kandi.csv",
        target="decade_rainfall",
        features=("tmax", "tmin", "insola", "weedpred", "humidR", "ETP"),
        year_column="Year",
        look_back=36,          # 36 dekads = one annual cycle
        unit="mm/dekad",
    ),
    "monthly": DatasetSpec(
        name="monthly",
        filename="Data_Kandi_end.csv",
        target="Rainfall_Amount",
        features=("Hmax", "Insola", "Vapeur_tension", "Vitess_vent", "Tmax", "Tmin"),
        # In Data_Kandi_end.csv the headers "Month" and "Year" are swapped:
        # the column labelled "Month" holds the calendar year (1970, 1971, ...).
        year_column="Month",
        look_back=12,          # 12 months = one annual cycle
        unit="mm/month",
    ),
}


@dataclass(frozen=True)
class HistoryConfig:
    """A historical-data configuration: full record and its train/test cut."""

    key: str
    first_year: int
    last_year: int
    train_last_year: int

    @property
    def label(self) -> str:
        n_train = self.train_last_year - self.first_year + 1
        return f"{n_train}-year training ({self.first_year}-{self.train_last_year})"


HISTORIES: dict[str, HistoryConfig] = {
    # 37-year record: train 1970-1999 (30 years), test 2000-2007.
    "37": HistoryConfig(key="37", first_year=1970, last_year=2007, train_last_year=1999),
    # 54-year record: train 1970-2013 (44 years), test 2014-2024.
    "54": HistoryConfig(key="54", first_year=1970, last_year=2024, train_last_year=2013),
}


## 2. Data: loading, split, imputation, scaling

In [ ]:
"""Loading, period selection, imputation, scaling and sample indexing.

Leakage rules (Methods, "Data preprocessing and input design"):
  * the KNN imputer and the scaler are fitted on training years only;
  * a training sample is kept only if *all* of its H targets fall in the
    training period, so no test observation is ever used as a training label;
  * test samples are those whose first target falls in the test period. Their
    input window may reach back into the training period (past values are
    legitimately known at forecast time).
The target is kept in its physical unit (mm); only the inputs are scaled.
"""


from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler



@dataclass
class PreparedData:
    """Everything the strategies need, for one dataset x configuration."""

    spec: DatasetSpec
    history: HistoryConfig
    inputs_scaled: np.ndarray   # (T, f) standardised inputs, target in column 0
    target: np.ndarray          # (T,)   rainfall in mm (after imputation)
    years: np.ndarray           # (T,)
    train_origins: np.ndarray   # indices t of the first target of each training sample
    test_origins: np.ndarray    # idem for test samples
    target_mean: float          # scaler parameters of column 0, used by the
    target_scale: float         # Recursive strategy to re-scale its forecasts
    n_imputed: int

    @property
    def look_back(self) -> int:
        return self.spec.look_back

    def scale_target(self, y_mm: np.ndarray | float) -> np.ndarray | float:
        return (y_mm - self.target_mean) / self.target_scale

    def summary(self) -> str:
        tr, te = self.train_origins, self.test_origins
        return (
            f"{self.spec.name} | {self.history.label} | "
            f"rows {self.years[0]}-{self.years[-1]} (T={len(self.years)}) | "
            f"train samples {len(tr)} (targets {self.years[tr[0]]}-{self.years[tr[-1] + HORIZON - 1]}) | "
            f"test samples {len(te)} (targets {self.years[te[0]]}-{self.years[te[-1] + HORIZON - 1]}) | "
            f"imputed values {self.n_imputed}"
        )


def load_raw(spec: DatasetSpec, data_dir: str | Path) -> pd.DataFrame:
    """Read the CSV file and keep the year column and the model inputs."""
    path = Path(data_dir) / spec.filename
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found. See data/README.md for the expected files and columns."
        )
    df = pd.read_csv(path, sep=spec.sep, decimal=spec.decimal)
    df.columns = df.columns.str.strip()
    missing = [c for c in (spec.year_column, *spec.inputs) if c not in df.columns]
    if missing:
        raise KeyError(f"{spec.filename}: missing column(s) {missing}; found {list(df.columns)}")
    df = df[[spec.year_column, *spec.inputs]].copy()
    df[spec.year_column] = pd.to_numeric(df[spec.year_column], errors="raise").astype(int)
    for col in spec.inputs:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df.reset_index(drop=True)


def prepare(
    spec: DatasetSpec,
    history: HistoryConfig,
    data_dir: str | Path | None = None,
    frame: pd.DataFrame | None = None,
    horizon: int = HORIZON,
) -> PreparedData:
    """Select the period, impute, scale and index the supervised samples.

    Pass either ``data_dir`` (the CSV is read) or an already loaded ``frame``
    with the columns of :func:`load_raw` (used by the tests).
    """
    df = load_raw(spec, data_dir) if frame is None else frame.copy()
    years_all = df[spec.year_column].to_numpy()
    keep = (years_all >= history.first_year) & (years_all <= history.last_year)
    df = df.loc[keep].reset_index(drop=True)
    years = df[spec.year_column].to_numpy()

    is_train_row = years <= history.train_last_year
    if is_train_row.all() or not is_train_row.any():
        raise ValueError(f"{spec.name}: no train/test split possible for {history}")

    raw = df[list(spec.inputs)].to_numpy(dtype=float)
    n_missing = int(np.isnan(raw).sum())

    imputer = KNNImputer(n_neighbors=KNN_NEIGHBORS).fit(raw[is_train_row])
    filled = imputer.transform(raw)
    scaler = StandardScaler().fit(filled[is_train_row])
    inputs_scaled = scaler.transform(filled)

    w, n = spec.look_back, len(years)
    first_test_row = int(np.argmax(~is_train_row))
    origins = np.arange(w, n - horizon + 1)            # t such that rows t-w..t+H-1 exist
    train_origins = origins[origins + horizon - 1 < first_test_row]
    test_origins = origins[origins >= first_test_row]

    return PreparedData(
        spec=spec,
        history=history,
        inputs_scaled=inputs_scaled,
        target=filled[:, 0],
        years=years,
        train_origins=train_origins,
        test_origins=test_origins,
        target_mean=float(scaler.mean_[0]),
        target_scale=float(scaler.scale_[0]),
        n_imputed=n_missing,
    )


## 3. Sliding windows

In [ ]:
"""Sliding-window construction of the supervised samples.

For an origin t (index of the first forecast step) and look-back w:
    X_t = inputs[t-w : t]          shape (w, f)
    Y_t = target[t : t+H]          shape (H,)
which is the X = [x_{t-w}, ..., x_{t-1}], Y = [y_t, ..., y_{t+H-1}] mapping of
the article.
"""


import numpy as np


def make_windows(
    inputs: np.ndarray,
    target: np.ndarray,
    origins: np.ndarray,
    look_back: int,
    horizon: int,
) -> tuple[np.ndarray, np.ndarray]:
    """Return X of shape (n, w, f) and Y of shape (n, H)."""
    origins = np.asarray(origins, dtype=int)
    if origins.min() < look_back or origins.max() + horizon > len(target):
        raise IndexError("an origin has an incomplete input window or target horizon")
    X = np.stack([inputs[t - look_back:t] for t in origins])
    Y = np.stack([target[t:t + horizon] for t in origins])
    return X, Y


def flatten(X: np.ndarray) -> np.ndarray:
    """(n, w, f) -> (n, w*f), used by every model except the LSTM."""
    return X.reshape(len(X), -1)


## 4. Metrics

In [ ]:
"""Performance indices (Methods, "Model performance evaluation").

Note: with the article's definitions, R^2 and NSE are the same quantity
(1 - SSE/SST). Both are reported, as in the article.
"""


import numpy as np


def rmse(obs: np.ndarray, sim: np.ndarray) -> float:
    return float(np.sqrt(np.mean((obs - sim) ** 2)))


def mae(obs: np.ndarray, sim: np.ndarray) -> float:
    return float(np.mean(np.abs(obs - sim)))


def pearson_r(obs: np.ndarray, sim: np.ndarray) -> float:
    if np.std(obs) == 0 or np.std(sim) == 0:
        return float("nan")
    return float(np.corrcoef(obs, sim)[0, 1])


def nse(obs: np.ndarray, sim: np.ndarray) -> float:
    """Nash-Sutcliffe efficiency."""
    return float(1.0 - np.sum((obs - sim) ** 2) / np.sum((obs - np.mean(obs)) ** 2))


def r2(obs: np.ndarray, sim: np.ndarray) -> float:
    """Coefficient of determination, 1 - SSE/SST (identical to NSE)."""
    return nse(obs, sim)


def kge(obs: np.ndarray, sim: np.ndarray) -> float:
    """Kling-Gupta efficiency (Gupta et al., 2009)."""
    r = pearson_r(obs, sim)
    alpha = np.std(sim) / np.std(obs)
    beta = np.mean(sim) / np.mean(obs)
    return float(1.0 - np.sqrt((r - 1.0) ** 2 + (alpha - 1.0) ** 2 + (beta - 1.0) ** 2))


METRICS = {"RMSE": rmse, "MAE": mae, "R": pearson_r, "R2": r2, "NSE": nse, "KGE": kge}


def evaluate(Y_obs: np.ndarray, Y_sim: np.ndarray) -> list[dict[str, float]]:
    """One row of metrics per horizon (columns of Y)."""
    Y_obs, Y_sim = np.atleast_2d(Y_obs), np.atleast_2d(Y_sim)
    if Y_obs.shape != Y_sim.shape:
        raise ValueError(f"shape mismatch {Y_obs.shape} vs {Y_sim.shape}")
    rows = []
    for h in range(Y_obs.shape[1]):
        row = {"horizon": f"t+{h + 1}"}
        row.update({name: fn(Y_obs[:, h], Y_sim[:, h]) for name, fn in METRICS.items()})
        rows.append(row)
    return rows


## 5. The six models

In [ ]:
"""The six forecasting models, their search spaces and reported hyperparameters.

All models share one small interface so that the strategies stay generic:

    model = build_model(name, params, seed)
    model.fit(X, Y, X_val=None, Y_val=None)   # Y has shape (n, k), k >= 1
    model.predict(X) -> array of shape (n, k)

``model.input_rank`` is 3 for the LSTM (inputs (n, w, f)) and 2 otherwise
(flattened windows (n, w*f)).

SVR, RF and CatBoost handle several outputs with one independent regressor
per output, which is what scikit-learn's ``MultiOutputRegressor`` does (see
Methods). The wrapper below is equivalent but also forwards a per-output
validation set, needed for CatBoost early stopping during tuning.

TensorFlow and CatBoost are imported lazily, so the classical models can be
used without them.
"""


from typing import Any

import numpy as np
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

# --------------------------------------------------------------------------
# Search spaces (Tables "Final optimized hyperparameter values", columns
# "Search Range") and search type ("grid" = exhaustive, "random" = sampled).
# --------------------------------------------------------------------------
SEARCH_SPACES: dict[str, dict[str, list[Any]]] = {
    "SVR": {
        "kernel": ["rbf", "poly", "linear"],
        "C": [1, 10, 50, 100],
        "epsilon": [0.01, 0.05, 0.1, 0.2],
        "gamma": [0.001, 0.01, 0.1],
    },
    "RF": {
        "n_estimators": [50, 100, 200, 500],
        "max_depth": [None, 5, 10, 15, 20],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
    },
    "CatBoost": {
        "iterations": [100, 200, 500, 1000],
        "depth": [4, 6, 8, 10],
        "learning_rate": [0.01, 0.03, 0.05, 0.1],
        "l2_leaf_reg": [1, 3, 5, 10],
        "early_stopping_rounds": [20, 50, 100],
    },
    "ANN": {
        "learning_rate": [0.0001, 0.001, 0.01],
        "dropout": [0.1, 0.2, 0.3, 0.4],
        "batch_size": [16, 32, 64, 128],
        "epochs": [50, 60, 100, 200],
    },
    "LSTM": {
        "units": [32, 64, 128, 256],
        "dropout": [0.1, 0.2, 0.3],
        "learning_rate": [0.0005, 0.001, 0.01],
        "epochs": [50, 100, 150, 200],
    },
    "TNN": {
        "hidden_units": [64, 128, 256, 512],
        "learning_rate": [0.0001, 0.0005, 0.005],
        "epochs": [50, 100, 150, 200],
    },
}

SEARCH_TYPE = {
    "SVR": "grid",       # GridSearchCV
    "RF": "random",      # RandomizedSearchCV, 50 iterations
    "CatBoost": "random",
    "ANN": "grid",       # structured search over the predefined grid
    "LSTM": "grid",
    "TNN": "grid",
}
RANDOM_SEARCH_ITER = 50

# --------------------------------------------------------------------------
# Hyperparameters reported in the article (columns "Dekadal" / "Monthly").
# Used when the experiment is run with --params reported (no tuning).
# --------------------------------------------------------------------------
REPORTED: dict[str, dict[str, dict[str, Any]]] = {
    "SVR": {
        "dekadal": {"kernel": "rbf", "C": 50, "epsilon": 0.1, "gamma": 0.01},
        "monthly": {"kernel": "rbf", "C": 50, "epsilon": 0.1, "gamma": 0.01},
    },
    "RF": {
        "dekadal": {"n_estimators": 200, "max_depth": 15, "min_samples_split": 2, "min_samples_leaf": 1},
        "monthly": {"n_estimators": 500, "max_depth": 15, "min_samples_split": 2, "min_samples_leaf": 1},
    },
    "CatBoost": {
        "dekadal": {"iterations": 200, "depth": 8, "learning_rate": 0.03, "l2_leaf_reg": 3,
                    "early_stopping_rounds": 50},
        "monthly": {"iterations": 500, "depth": 8, "learning_rate": 0.03, "l2_leaf_reg": 3,
                    "early_stopping_rounds": 50},
    },
    "ANN": {
        "dekadal": {"learning_rate": 0.001, "dropout": 0.3, "batch_size": 32, "epochs": 60},
        "monthly": {"learning_rate": 0.001, "dropout": 0.2, "batch_size": 64, "epochs": 50},
    },
    "LSTM": {
        "dekadal": {"units": 64, "dropout": 0.2, "learning_rate": 0.001, "epochs": 150},
        "monthly": {"units": 128, "dropout": 0.2, "learning_rate": 0.0005, "epochs": 100},
    },
    "TNN": {
        "dekadal": {"hidden_units": 256, "learning_rate": 0.0005, "epochs": 200},
        "monthly": {"hidden_units": 128, "learning_rate": 0.005, "epochs": 100},
    },
}

# Fixed settings that are not tuned (Table "Fixed architecture design").
LSTM_BATCH_SIZE = 32
LSTM_DENSE_UNITS = 32
TNN_BATCH_SIZE = 32
TNN_DROPOUT = 0.3
ANN_LAYERS = (256, 128, 64)


# ==========================================================================
# Classical models
# ==========================================================================
class IndependentOutputs:
    """One clone of ``base`` per output column (= MultiOutputRegressor)."""

    input_rank = 2

    def __init__(self, base, uses_eval_set: bool = False):
        self.base = base
        self.uses_eval_set = uses_eval_set
        self.estimators_: list = []

    def fit(self, X, Y, X_val=None, Y_val=None):
        Y = np.asarray(Y).reshape(len(X), -1)
        Y_val = None if Y_val is None else np.asarray(Y_val).reshape(len(X_val), -1)
        self.estimators_ = []
        for j in range(Y.shape[1]):
            est = clone(self.base)
            if self.uses_eval_set and X_val is not None:
                est.fit(X, Y[:, j], eval_set=(X_val, Y_val[:, j]))
            else:
                est.fit(X, Y[:, j])
            self.estimators_.append(est)
        return self

    def predict(self, X):
        return np.column_stack([est.predict(X) for est in self.estimators_])


def _svr(params: dict, seed: int) -> IndependentOutputs:
    return IndependentOutputs(SVR(**params))


def _random_forest(params: dict, seed: int) -> IndependentOutputs:
    return IndependentOutputs(RandomForestRegressor(random_state=seed, n_jobs=-1, **params))


def _catboost(params: dict, seed: int) -> IndependentOutputs:
    from catboost import CatBoostRegressor

    p = dict(params)
    early = p.pop("early_stopping_rounds", None)
    base = CatBoostRegressor(
        loss_function="RMSE",
        random_seed=seed,
        verbose=0,
        allow_writing_files=False,
        thread_count=-1,
        early_stopping_rounds=early,   # only active when an eval_set is given (tuning folds)
        **p,
    )
    return IndependentOutputs(base, uses_eval_set=True)


# ==========================================================================
# Neural models (TensorFlow / Keras)
# ==========================================================================
class KerasModel:
    """Builds a fresh, seeded Keras network at each call of ``fit``.

    Training follows the article: no internal validation split, samples
    presented in chronological order (shuffle=False), target in mm.
    """

    def __init__(self, kind: str, params: dict, seed: int):
        self.kind = kind
        self.params = dict(params)
        self.seed = seed
        self.input_rank = 3 if kind == "LSTM" else 2
        self.net_ = None

    def _build(self, input_shape: tuple[int, ...], n_outputs: int):
        from tensorflow.keras import layers, models, optimizers

        p = self.params
        inp = layers.Input(shape=input_shape)
        if self.kind == "ANN":
            x = inp
            for i, units in enumerate(ANN_LAYERS):
                x = layers.Dense(units, activation="relu")(x)
                if i < len(ANN_LAYERS) - 1:
                    x = layers.Dropout(p["dropout"])(x)
            loss = "mae"
        elif self.kind == "LSTM":
            x = layers.LSTM(p["units"], activation="tanh",
                            dropout=p["dropout"], recurrent_dropout=p["dropout"])(inp)
            x = layers.Dropout(p["dropout"])(x)
            x = layers.Dense(LSTM_DENSE_UNITS, activation="relu")(x)
            loss = "mse"
        elif self.kind == "TNN":
            h = p["hidden_units"]
            x = layers.Dense(h, activation="relu")(inp)
            x = layers.Dropout(TNN_DROPOUT)(x)
            x = layers.Dense(h // 2, activation="relu")(x)
            x = layers.Dropout(TNN_DROPOUT)(x)
            x = layers.Dense(h // 4, activation="relu")(x)
            loss = "mse"
        else:
            raise ValueError(self.kind)
        out = layers.Dense(n_outputs, activation="linear")(x)
        net = models.Model(inp, out)
        net.compile(optimizer=optimizers.Adam(learning_rate=p["learning_rate"]), loss=loss)
        return net

    def _batch_size(self) -> int:
        if self.kind == "ANN":
            return int(self.params["batch_size"])
        return LSTM_BATCH_SIZE if self.kind == "LSTM" else TNN_BATCH_SIZE

    def fit(self, X, Y, X_val=None, Y_val=None):
        import tensorflow as tf

        tf.keras.utils.set_random_seed(self.seed)
        Y = np.asarray(Y, dtype="float32").reshape(len(X), -1)
        self.net_ = self._build(X.shape[1:], Y.shape[1])
        self.net_.fit(
            np.asarray(X, dtype="float32"), Y,
            epochs=int(self.params["epochs"]),
            batch_size=self._batch_size(),
            shuffle=False,
            verbose=0,
        )
        return self

    def predict(self, X):
        return self.net_.predict(np.asarray(X, dtype="float32"), verbose=0)


# ==========================================================================
_BUILDERS = {
    "SVR": _svr,
    "RF": _random_forest,
    "CatBoost": _catboost,
    "ANN": lambda params, seed: KerasModel("ANN", params, seed),
    "LSTM": lambda params, seed: KerasModel("LSTM", params, seed),
    "TNN": lambda params, seed: KerasModel("TNN", params, seed),
}


def build_model(name: str, params: dict, seed: int):
    try:
        return _BUILDERS[name](dict(params), seed)
    except KeyError:
        raise ValueError(f"unknown model {name!r}; choose from {list(_BUILDERS)}") from None


def input_rank(name: str) -> int:
    return 3 if name == "LSTM" else 2


## 6. Hyperparameter search (TSCV)

In [ ]:
"""Hyperparameter search with time-series cross-validation (TSCV).

Each candidate is scored by the mean, over the TimeSeriesSplit folds of the
training samples, of the RMSE on the validation fold (all outputs pooled).
Folds are chronological: every validation fold lies after its training fold,
so no future information is used. The test period is never touched here.
"""


from dataclasses import dataclass, field
from typing import Any

import numpy as np
from sklearn.model_selection import ParameterGrid, ParameterSampler, TimeSeriesSplit



@dataclass
class TuningResult:
    best_params: dict[str, Any]
    best_score: float
    table: list[dict[str, Any]] = field(default_factory=list)   # every candidate and its score


def candidates(model_name: str, seed: int, max_candidates: int | None = None) -> list[dict]:
    space = SEARCH_SPACES[model_name]
    if SEARCH_TYPE[model_name] == "grid":
        cands = list(ParameterGrid(space))
    else:
        cands = list(ParameterSampler(space, n_iter=RANDOM_SEARCH_ITER, random_state=seed))
    if max_candidates is not None and len(cands) > max_candidates:
        rng = np.random.default_rng(seed)
        cands = [cands[i] for i in sorted(rng.choice(len(cands), max_candidates, replace=False))]
    return cands


def tscv_search(
    model_name: str,
    X: np.ndarray,
    Y: np.ndarray,
    seed: int,
    n_splits: int,
    max_candidates: int | None = None,
) -> TuningResult:
    """Return the candidate with the lowest mean validation RMSE."""
    Y = np.asarray(Y).reshape(len(X), -1)
    splitter = TimeSeriesSplit(n_splits=n_splits)
    table = []
    for params in candidates(model_name, seed, max_candidates):
        scores = []
        for tr, va in splitter.split(X):
            model = build_model(model_name, params, seed)
            model.fit(X[tr], Y[tr], X_val=X[va], Y_val=Y[va])
            pred = np.asarray(model.predict(X[va])).reshape(len(va), -1)
            scores.append(float(np.sqrt(np.mean((Y[va] - pred) ** 2))))
        table.append({**params, "cv_rmse": float(np.mean(scores))})
    best = min(table, key=lambda row: row["cv_rmse"])
    best_params = {k: v for k, v in best.items() if k != "cv_rmse"}
    return TuningResult(best_params=best_params, best_score=best["cv_rmse"], table=table)


## 7. The four strategies

In [ ]:
"""The four multi-step strategies: Direct, Recursive, MIMO and DIRMO.

A strategy is a set of sub-models, each trained on some columns of the target
matrix Y = [y_{t}, y_{t+1}, y_{t+2}] (H = 3):

    Direct     3 models: {T+1}, {T+2}, {T+3}
    MIMO       1 model : {T+1, T+2, T+3}
    DIRMO      2 models: {T+1, T+2}, {T+3}          (blocks of config.DIRMO_BLOCKS)
    Recursive  1 model : {T+1}, applied iteratively

Recursive forecasting. After predicting y_{t+h}, the input window is shifted
by one step and the new last row is filled with
  * the forecast y_{t+h} (re-scaled) in the rainfall column, and
  * the exogenous predictors of the same time step, x_{t+h},
which is the recursion  y_{t+h} = f(y_{t+h-1}, ..., x_{t+h-1})  of the article.
Using observed exogenous values assumes they are known (perfect-prognosis
setting). ``exog_mode="persistence"`` instead repeats the last observed
exogenous row, an operational setting where future predictors are unknown.
"""


from dataclasses import dataclass, field
from typing import Callable

import numpy as np


ParamsFn = Callable[[str, np.ndarray, np.ndarray], dict]
"""(sub-model key, X_train, Y_train_sub) -> hyperparameters for that sub-model."""


def submodels(strategy: str, horizon: int = HORIZON) -> list[tuple[str, list[int]]]:
    if strategy == "Direct":
        return [(f"t+{h + 1}", [h]) for h in range(horizon)]
    if strategy == "MIMO":
        return [("t+1..t+%d" % horizon, list(range(horizon)))]
    if strategy == "DIRMO":
        return [(f"block{i + 1}", list(range(a, b))) for i, (a, b) in enumerate(DIRMO_BLOCKS)]
    if strategy == "Recursive":
        return [("one-step", [0])]
    raise ValueError(f"unknown strategy {strategy!r}")


@dataclass
class StrategyOutput:
    Y_true: np.ndarray                  # (n_test, H) in mm
    Y_pred: np.ndarray                  # (n_test, H) in mm
    params: dict[str, dict] = field(default_factory=dict)   # hyperparameters per sub-model


def _model_input(X3: np.ndarray, rank: int) -> np.ndarray:
    return X3 if rank == 3 else flatten(X3)


def run_strategy(
    strategy: str,
    model_name: str,
    data: PreparedData,
    params_fn: ParamsFn,
    seed: int,
    exog_mode: str = "observed",
    horizon: int = HORIZON,
) -> StrategyOutput:
    w, rank = data.look_back, input_rank(model_name)
    X3_tr, Y_tr = make_windows(data.inputs_scaled, data.target, data.train_origins, w, horizon)
    X3_te, Y_te = make_windows(data.inputs_scaled, data.target, data.test_origins, w, horizon)
    X_tr, X_te = _model_input(X3_tr, rank), _model_input(X3_te, rank)

    Y_pred = np.full_like(Y_te, np.nan, dtype=float)
    used: dict[str, dict] = {}
    for key, cols in submodels(strategy, horizon):
        params = params_fn(key, X_tr, Y_tr[:, cols])
        model = build_model(model_name, params, seed).fit(X_tr, Y_tr[:, cols])
        used[key] = params
        if strategy == "Recursive":
            Y_pred = _recursive_forecast(model, data, data.test_origins, rank, exog_mode, horizon)
        else:
            Y_pred[:, cols] = np.asarray(model.predict(X_te)).reshape(len(X_te), -1)

    if np.isnan(Y_pred).any():
        raise RuntimeError(f"{strategy}: some horizons were not predicted")
    return StrategyOutput(Y_true=Y_te, Y_pred=Y_pred, params=used)


def _recursive_forecast(
    model,
    data: PreparedData,
    origins: np.ndarray,
    rank: int,
    exog_mode: str,
    horizon: int,
) -> np.ndarray:
    """Iterate a one-step model over the horizon, for all origins at once."""
    if exog_mode not in ("observed", "persistence"):
        raise ValueError("exog_mode must be 'observed' or 'persistence'")
    w, S = data.look_back, data.inputs_scaled
    windows = np.stack([S[t - w:t] for t in origins]).copy()      # (n, w, f)
    preds = np.empty((len(origins), horizon))
    for h in range(horizon):
        y_hat = np.asarray(model.predict(_model_input(windows, rank))).reshape(-1)
        preds[:, h] = y_hat
        if h == horizon - 1:
            break
        if exog_mode == "observed":
            new_rows = S[origins + h].copy()          # exogenous values at time t+h
        else:
            new_rows = windows[:, -1, :].copy()       # last known exogenous values
        new_rows[:, 0] = data.scale_target(y_hat)     # rainfall column <- forecast
        windows = np.concatenate([windows[:, 1:, :], new_rows[:, None, :]], axis=1)
    return preds


## 8. One experimental run

In [ ]:
"""One experimental run = dataset x historical configuration x model x strategy."""


import json
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd



def set_global_seed(seed: int = SEED) -> None:
    """Seed Python, NumPy and (if installed) TensorFlow."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        import tensorflow as tf
    except ImportError:
        return
    tf.keras.utils.set_random_seed(seed)


def run_one(
    data: PreparedData,
    model: str,
    strategy: str,
    params_mode: str = "reported",
    seed: int = SEED,
    n_splits: int = TSCV_SPLITS,
    max_candidates: int | None = None,
    exog_mode: str = "observed",
    out_dir: str | Path | None = None,
) -> pd.DataFrame:
    """Train, forecast the test period and return one metrics row per horizon.

    params_mode="reported": hyperparameters of the article's tables.
    params_mode="tune":     TSCV search on the training samples, separately for
                            every sub-model of the strategy.
    """
    resolution, history = data.spec.name, data.history.key
    tuning_tables: dict[str, list] = {}

    def params_fn(key: str, X: np.ndarray, Y: np.ndarray) -> dict:
        if params_mode == "reported":
            return dict(REPORTED[model][resolution])
        if params_mode == "tune":
            res = tscv_search(model, X, Y, seed=seed, n_splits=n_splits, max_candidates=max_candidates)
            tuning_tables[key] = res.table
            return res.best_params
        raise ValueError("params_mode must be 'reported' or 'tune'")

    set_global_seed(seed)
    out = run_strategy(strategy, model, data, params_fn, seed=seed, exog_mode=exog_mode)

    rows = pd.DataFrame(evaluate(out.Y_true, out.Y_pred))
    rows.insert(0, "strategy", strategy)
    rows.insert(0, "model", model)
    rows.insert(0, "history", history)
    rows.insert(0, "resolution", resolution)

    if out_dir is not None:
        run_dir = Path(out_dir) / resolution / f"{history}y" / f"{model}_{strategy}"
        run_dir.mkdir(parents=True, exist_ok=True)
        origins = data.test_origins
        pred = pd.DataFrame({"origin_index": origins, "origin_year": data.years[origins]})
        for h in range(HORIZON):
            pred[f"obs_t+{h + 1}"] = out.Y_true[:, h]
            pred[f"pred_t+{h + 1}"] = out.Y_pred[:, h]
        pred.to_csv(run_dir / "predictions.csv", index=False)
        rows.to_csv(run_dir / "metrics.csv", index=False)
        (run_dir / "hyperparameters.json").write_text(
            json.dumps({"mode": params_mode, "seed": seed, "submodels": out.params}, indent=2, default=str)
        )
        if tuning_tables:
            (run_dir / "tuning.json").write_text(json.dumps(tuning_tables, indent=2, default=str))
    return rows


## 9. Statistical tests

In [ ]:
"""Non-parametric comparison of forecasting errors (Demšar, 2006).

Input: the long table written by ``scripts/run_experiments.py`` with columns
resolution, history, model, strategy, horizon, MAE (one row per run and horizon).

* Friedman test across strategies, blocks = (model, horizon, history);
* Friedman test across models,     blocks = (strategy, horizon, history);
* Nemenyi post-hoc test when the Friedman test is significant;
* paired Wilcoxon signed-rank test, 37-year vs 54-year record,
  pairs = (model, strategy, horizon).
All tests are run separately for each temporal resolution.
"""


from dataclasses import dataclass

import numpy as np
import pandas as pd
from scipy import stats


@dataclass
class FriedmanResult:
    statistic: float
    p_value: float
    n_blocks: int
    mean_ranks: pd.Series


def _blocks(df: pd.DataFrame, factor: str, block_cols: list[str], value: str) -> pd.DataFrame:
    table = df.pivot_table(index=block_cols, columns=factor, values=value, aggfunc="first")
    return table.dropna(axis=0, how="any")      # keep only complete blocks (matched design)


def friedman(df: pd.DataFrame, factor: str, block_cols: list[str], value: str = "MAE") -> FriedmanResult:
    table = _blocks(df, factor, block_cols, value)
    if table.shape[1] < 3:
        raise ValueError("the Friedman test needs at least three groups")
    stat, p = stats.friedmanchisquare(*[table[c].to_numpy() for c in table.columns])
    ranks = table.rank(axis=1).mean(axis=0).sort_values()
    return FriedmanResult(float(stat), float(p), len(table), ranks)


def nemenyi(df: pd.DataFrame, factor: str, block_cols: list[str], value: str = "MAE") -> pd.DataFrame:
    """Matrix of Nemenyi p-values (two-sided), from mean ranks within blocks."""
    table = _blocks(df, factor, block_cols, value)
    n, k = table.shape
    mean_ranks = table.rank(axis=1).mean(axis=0)
    se = np.sqrt(k * (k + 1) / (6.0 * n))
    groups = list(table.columns)
    p = pd.DataFrame(np.ones((k, k)), index=groups, columns=groups)
    for i, a in enumerate(groups):
        for j, b in enumerate(groups):
            if i < j:
                q = abs(mean_ranks[a] - mean_ranks[b]) / se
                p_ij = float(stats.studentized_range.sf(q * np.sqrt(2.0), k, np.inf))
                p.loc[a, b] = p.loc[b, a] = min(1.0, p_ij)
    return p


def wilcoxon_histories(
    df: pd.DataFrame,
    pair_cols: tuple[str, ...] = ("model", "strategy", "horizon"),
    value: str = "MAE",
    short: str = "37",
    long: str = "54",
) -> dict[str, float]:
    df = df.assign(history=df["history"].astype(str))
    table = df.pivot_table(index=list(pair_cols), columns="history", values=value, aggfunc="first")
    table = table[[short, long]].dropna()
    res = stats.wilcoxon(table[short], table[long], alternative="two-sided")
    return {
        "n_pairs": int(len(table)),
        "statistic": float(res.statistic),
        "p_value": float(res.pvalue),
        "median_difference": float(np.median(table[short] - table[long])),
    }


## 10. Figures

In [ ]:
"""Figures: rainfall trend (Fig 1) and observed-vs-forecast series."""


from pathlib import Path

import matplotlib

import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from scipy import stats  # noqa: E402



def rainfall_trend(df: pd.DataFrame, spec: DatasetSpec, out: str | Path, title: str) -> Path:
    """Annual mean of the dekadal (or monthly) rainfall with its linear trend."""
    annual = df.groupby(spec.year_column)[spec.target].mean()
    x, y = annual.index.to_numpy(float), annual.to_numpy(float)
    fit = stats.linregress(x, y)

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(x, y, marker="o", lw=2, color="darkblue", label=f"{spec.name.capitalize()} rainfall")
    ax.plot(x, fit.intercept + fit.slope * x, color="red", lw=2,
            label=f"Trend ({fit.slope:+.3f} mm/yr, p = {fit.pvalue:.3f})")
    ax.set(title=title, xlabel="Year", ylabel=f"{spec.name.capitalize()} rainfall ({spec.unit.split('/')[0]})")
    ax.grid(True, ls="--", alpha=0.6)
    ax.legend()
    fig.tight_layout()
    out = Path(out)
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return out


def observed_vs_forecast(Y_true: np.ndarray, Y_pred: np.ndarray, title: str, ylabel: str,
                         out: str | Path) -> Path:
    """One panel per horizon."""
    H = Y_true.shape[1]
    fig, axes = plt.subplots(H, 1, figsize=(10, 3 * H), sharex=True)
    for h, ax in enumerate(np.atleast_1d(axes)):
        ax.plot(Y_true[:, h], color="black", lw=1.2, label="Observed")
        ax.plot(Y_pred[:, h], color="tab:blue", lw=1.2, label="Forecast")
        ax.set_title(f"t+{h + 1}", loc="left")
        ax.set_ylabel(ylabel)
        ax.legend(loc="upper right")
    axes[-1].set_xlabel("Test sample (chronological)")
    fig.suptitle(title)
    fig.tight_layout()
    out = Path(out)
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return out


## 11. Check the data

In [ ]:
data = prepare(DATASETS["monthly"], HISTORIES["54"], data_dir=DATA_DIR)
print(data.summary())

## 12. Quick test: SVR and RF, monthly, 54-year record

In [ ]:
rows = []
for model in ["SVR", "RF"]:
    for strategy in STRATEGIES:
        rows.append(run_one(data, model, strategy, out_dir=OUT_DIR))
pd.concat(rows, ignore_index=True)

## 13. Test CatBoost and the neural networks (Direct strategy)

In [ ]:
for model in ["CatBoost", "ANN", "LSTM", "TNN"]:
    display(run_one(data, model, "Direct", out_dir=OUT_DIR))

## 14. All experiments (2 resolutions × 2 records × 6 models × 4 strategies)

`params_mode="reported"` uses the hyperparameters of Tables 2 and 4; `"tune"` re-tunes them by time-series cross-validation (slow).

In [ ]:
PARAMS_MODE = "reported"

all_rows = []
for res in DATASETS:
    for hist in HISTORIES:
        d = prepare(DATASETS[res], HISTORIES[hist], data_dir=DATA_DIR)
        print(d.summary())
        for model in MODELS:
            for strategy in STRATEGIES:
                r = run_one(d, model, strategy, params_mode=PARAMS_MODE, out_dir=OUT_DIR)
                print(f"  {model:8s} {strategy:9s} MAE", r["MAE"].round(2).tolist())
                all_rows.append(r)
metrics = pd.concat(all_rows, ignore_index=True)
metrics.to_csv(f"{OUT_DIR}/metrics.csv", index=False)
metrics

## 15. Statistical tests

In [ ]:
for res, d in metrics.groupby("resolution"):
    for factor, blocks in (("strategy", ["model", "horizon", "history"]),
                           ("model", ["strategy", "horizon", "history"])):
        if d[factor].nunique() < 3:
            continue
        fr = friedman(d, factor, blocks)
        print(f"[{res}] Friedman across {factor} levels: chi2 = {fr.statistic:.3f}, p = {fr.p_value:.3g}")
        print("   mean ranks:", fr.mean_ranks.round(2).to_dict())
        if fr.p_value < 0.05:
            display(nemenyi(d, factor, blocks).round(4))
    wx = wilcoxon_histories(d)
    print(f"[{res}] Wilcoxon 37 vs 54 years: V = {wx['statistic']:.1f}, p = {wx['p_value']:.3g}, pairs = {wx['n_pairs']}")

## 16. Trend figure (Fig 1)

In [ ]:
spec = DATASETS["dekadal"]
df = load_raw(spec, DATA_DIR).dropna(subset=[spec.target])
rainfall_trend(df, spec, f"{OUT_DIR}/Fig1_trend.png",
               title=f"Kandi {df[spec.year_column].min()} - {df[spec.year_column].max()}")